# Pass 4 — Robustness to Imperfect ROI Localization

This inference-only experiment tests the frozen Pass 3B raw-ROI VGG16 classifier under imperfect lesion localization.

Design: locked Pass 3B test cohort; no retraining; no fine-tuning; threshold fixed at 0.5; perturbation levels 0%, ±5%, ±10%, ±15%; five reproducible perturbation replicates for each non-zero level. The perturbation changes the bounding box used to crop the dilated ground-truth lesion mask, while leaving the lesion mask itself unchanged. Thus this is a test of localization/cropping robustness, not end-to-end detection or segmentation robustness.

## Research question

**RQ2:** How sensitive is the trained lesion classifier to imperfections in lesion localization?

Primary metrics: ROC-AUC, PR-AUC, sensitivity, specificity, F1, balanced accuracy, and accuracy. The notebook also saves per-image predictions, box IoU, confusion matrices/metrics by repeat, bootstrap 95% CIs, and publication-ready degradation plots.

In [ ]:
from pathlib import Path
import os, random, json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix)
import tensorflow as tf

SEED=42
os.environ["PYTHONHASHSEED"]=str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE=(224,224); BATCH_SIZE=16
DATA_ROOT=Path("../data/Dataset_BUSI_with_GT")
PASS3B_DIR=Path("../results/pass3b_vgg16")
SPLIT_CSV=PASS3B_DIR/"split"/"Pass_3B_locked_split.csv"
RAW_ROI_MODEL=PASS3B_DIR/"models"/"vgg16_raw_roi.keras"
OUTPUT_DIR=Path("../results/pass4_localization_robustness")
PRED_DIR=OUTPUT_DIR/"predictions"; FIG_DIR=OUTPUT_DIR/"figures"
for d in [OUTPUT_DIR,PRED_DIR,FIG_DIR]: d.mkdir(parents=True,exist_ok=True)
PERTURBATION_LEVELS=[0.00,0.05,0.10,0.15]
N_REPEATS=5
BOOTSTRAP_N=2000
print("Split:",SPLIT_CSV); print("Model:",RAW_ROI_MODEL)
print("Split exists:",SPLIT_CSV.exists(),"Model exists:",RAW_ROI_MODEL.exists())

In [ ]:
if not SPLIT_CSV.exists(): raise FileNotFoundError(f"Locked Pass 3B split not found: {SPLIT_CSV}")
if not RAW_ROI_MODEL.exists(): raise FileNotFoundError(f"Pass 3B raw-ROI model not found: {RAW_ROI_MODEL}")
split_df=pd.read_csv(SPLIT_CSV)
required={"image_path","filename","class","label","split","md5","split_group_id"}
missing=required-set(split_df.columns)
if missing: raise ValueError(f"Missing columns: {sorted(missing)}")
test_df=split_df.loc[split_df["split"]=="test"].copy().reset_index(drop=True)
assert test_df["split_group_id"].is_unique
def resolve_original_path(row):
    p=Path(str(row["image_path"]))
    return p if p.exists() else DATA_ROOT/str(row["class"])/str(row["filename"])
def resolve_mask_path(image_path):
    p=Path(image_path); return p.with_name(f"{p.stem}_mask{p.suffix}")
test_df["image_resolved"]=test_df.apply(resolve_original_path,axis=1)
test_df["mask_path"]=test_df["image_resolved"].map(resolve_mask_path)
if (~test_df["image_resolved"].map(lambda p: Path(p).exists())).any(): raise FileNotFoundError("Missing test image(s).")
if (~test_df["mask_path"].map(lambda p: Path(p).exists())).any(): raise FileNotFoundError("Missing ground-truth mask(s).")
print("Test images:",len(test_df))

## 1. Reproduce the Pass 3B raw-ROI construction

The reference pipeline uses the binary ground-truth mask, a 15×15 elliptical dilation, masking, the tight non-zero bounding box, and 224×224 resizing. At 0% perturbation the Pass 4 input must exactly reproduce the Pass 3B raw ROI.

In [ ]:
KERNEL_SIZE=15
def load_rgb(path):
    image=cv2.imread(str(path))
    if image is None: raise ValueError(f"Could not load image: {path}")
    return cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
def load_binary_mask(mask_path,target_shape):
    mask=cv2.imread(str(mask_path),cv2.IMREAD_GRAYSCALE)
    if mask is None: raise ValueError(f"Could not load mask: {mask_path}")
    h,w=target_shape[:2]
    if mask.shape!=(h,w): mask=cv2.resize(mask,(w,h),interpolation=cv2.INTER_NEAREST)
    return (mask>0).astype(np.uint8)
def dilate_mask(mask,kernel_size=KERNEL_SIZE):
    kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(kernel_size,kernel_size))
    return cv2.dilate(mask,kernel,iterations=1)
def crop_box_from_mask(mask):
    ys,xs=np.where(mask>0)
    if len(xs)==0: return None
    return np.array([xs.min(),ys.min(),xs.max()+1,ys.max()+1],dtype=np.float32)
def crop_masked_image(image_rgb,mask,box):
    masked=image_rgb.copy(); masked[mask==0]=0
    h,w=masked.shape[:2]; x0,y0,x1,y1=box
    x0=int(np.clip(np.floor(x0),0,w-1)); y0=int(np.clip(np.floor(y0),0,h-1))
    x1=int(np.clip(np.ceil(x1),x0+1,w)); y1=int(np.clip(np.ceil(y1),y0+1,h))
    crop=masked[y0:y1,x0:x1]
    if crop.size==0: raise ValueError("Empty perturbed crop")
    return cv2.resize(crop,IMG_SIZE,interpolation=cv2.INTER_AREA)
def make_reference_roi(image_path,mask_path):
    image=load_rgb(image_path); mask=load_binary_mask(mask_path,image.shape); dilated=dilate_mask(mask); box=crop_box_from_mask(dilated)
    if box is None: raise ValueError(f"Empty lesion mask: {mask_path}")
    return crop_masked_image(image,dilated,box),dilated,box

In [ ]:
sample=test_df.iloc[0]
ref_roi,ref_mask,ref_box=make_reference_roi(sample["image_resolved"],sample["mask_path"])
zero_box=ref_box.copy(); zero_roi=crop_masked_image(load_rgb(sample["image_resolved"]),ref_mask,zero_box)
assert np.array_equal(ref_roi,zero_roi)
print("PASS: 0% perturbation exactly reproduces the reference raw ROI.")

## 2. Perturb the localization box

For perturbation level `p`, x/y centre shifts are sampled within ±p of the reference box dimensions, while width and height are independently scaled within [1−p, 1+p]. Boxes are clipped to the image boundaries. The lesion mask remains fixed; only the crop/localization box changes.

In [ ]:
def perturb_box(box,image_shape,level,rng):
    h,w=image_shape[:2]; x0,y0,x1,y1=map(float,box); bw=x1-x0; bh=y1-y0; cx=(x0+x1)/2; cy=(y0+y1)/2
    if level==0: return np.array([x0,y0,x1,y1],dtype=np.float32)
    dx=rng.uniform(-level,level)*bw; dy=rng.uniform(-level,level)*bh
    new_bw=max(2.0,bw*rng.uniform(1-level,1+level)); new_bh=max(2.0,bh*rng.uniform(1-level,1+level))
    ncx=cx+dx; ncy=cy+dy
    px0=np.clip(ncx-new_bw/2,0,w-1); py0=np.clip(ncy-new_bh/2,0,h-1)
    px1=np.clip(ncx+new_bw/2,px0+1,w); py1=np.clip(ncy+new_bh/2,py0+1,h)
    return np.array([px0,py0,px1,py1],dtype=np.float32)
def box_iou(a,b):
    ax0,ay0,ax1,ay1=a; bx0,by0,bx1,by1=b
    ix=max(0,min(ax1,bx1)-max(ax0,bx0)); iy=max(0,min(ay1,by1)-max(ay0,by0)); inter=ix*iy
    aa=max(0,ax1-ax0)*max(0,ay1-ay0); ab=max(0,bx1-bx0)*max(0,by1-by0); union=aa+ab-inter
    return inter/union if union else np.nan

## 3. Load the frozen Pass 3B raw-ROI model

No training, fine-tuning, validation-based selection, or threshold optimization occurs in Pass 4.

In [ ]:
model=tf.keras.models.load_model(RAW_ROI_MODEL)
print("Loaded:",RAW_ROI_MODEL)
print("Trainable parameters:",sum(np.prod(v.shape) for v in model.trainable_weights))
assert len(model.trainable_weights)==0

In [ ]:
def make_perturbed_dataset(frame,level,repeat,base_seed=SEED):
    rng=np.random.default_rng(base_seed+repeat+int(level*10000)); X=[]; rows=[]
    for _,row in frame.iterrows():
        image=load_rgb(row["image_resolved"]); mask=load_binary_mask(row["mask_path"],image.shape); dilated=dilate_mask(mask); ref=crop_box_from_mask(dilated)
        if ref is None: raise ValueError(f"Empty lesion mask: {row['filename']}")
        pbox=perturb_box(ref,image.shape,level,rng); roi=crop_masked_image(image,dilated,pbox)
        X.append(roi.astype(np.float32)/255.0)
        rows.append({"filename":row["filename"],"class":row["class"],"label":int(row["label"]),"split_group_id":row["split_group_id"],"perturbation_level":level,"repeat":repeat,"reference_x0":ref[0],"reference_y0":ref[1],"reference_x1":ref[2],"reference_y1":ref[3],"perturbed_x0":pbox[0],"perturbed_y0":pbox[1],"perturbed_x1":pbox[2],"perturbed_y1":pbox[3],"box_iou":box_iou(ref,pbox)})
    return np.stack(X),pd.DataFrame(rows)
all_prediction_rows=[]; all_metric_rows=[]
for level in PERTURBATION_LEVELS:
    repeats=1 if level==0 else N_REPEATS
    for repeat in range(repeats):
        X,meta=make_perturbed_dataset(test_df,level,repeat)
        prob=model.predict(X,batch_size=BATCH_SIZE,verbose=0).ravel(); pred=(prob>=0.5).astype(int); y=meta["label"].to_numpy()
        tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
        all_metric_rows.append({"perturbation_level":level,"perturbation_percent":int(level*100),"repeat":repeat,"n_test":len(y),"accuracy":accuracy_score(y,pred),"balanced_accuracy":balanced_accuracy_score(y,pred),"precision":precision_score(y,pred,zero_division=0),"sensitivity":recall_score(y,pred,zero_division=0),"specificity":tn/(tn+fp) if tn+fp else np.nan,"f1":f1_score(y,pred,zero_division=0),"roc_auc":roc_auc_score(y,prob),"pr_auc":average_precision_score(y,prob),"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),"mean_box_iou":meta["box_iou"].mean(),"median_box_iou":meta["box_iou"].median()})
        meta["predicted_probability_malignant"]=prob; meta["predicted_label"]=pred; meta["predicted_class"]=np.where(pred==1,"malignant","benign"); all_prediction_rows.append(meta)
metrics_df=pd.DataFrame(all_metric_rows); predictions_df=pd.concat(all_prediction_rows,ignore_index=True)
metrics_df.to_csv(OUTPUT_DIR/"pass4_localization_robustness_metrics_by_repeat.csv",index=False)
predictions_df.to_csv(PRED_DIR/"pass4_localization_robustness_predictions.csv",index=False)
display(metrics_df.round(4))

## 4. Aggregate results across perturbation replicates

In [ ]:
summary_metrics=(metrics_df.groupby(["perturbation_level","perturbation_percent"],as_index=False).agg(accuracy_mean=("accuracy","mean"),accuracy_sd=("accuracy","std"),balanced_accuracy_mean=("balanced_accuracy","mean"),balanced_accuracy_sd=("balanced_accuracy","std"),sensitivity_mean=("sensitivity","mean"),sensitivity_sd=("sensitivity","std"),specificity_mean=("specificity","mean"),specificity_sd=("specificity","std"),f1_mean=("f1","mean"),f1_sd=("f1","std"),roc_auc_mean=("roc_auc","mean"),roc_auc_sd=("roc_auc","std"),pr_auc_mean=("pr_auc","mean"),pr_auc_sd=("pr_auc","std"),mean_box_iou=("mean_box_iou","mean"),mean_box_iou_sd=("mean_box_iou","std")))
summary_metrics.to_csv(OUTPUT_DIR/"pass4_localization_robustness_summary.csv",index=False)
display(summary_metrics.round(4))

## 5. Bootstrap 95% confidence intervals

Bootstrap uncertainty is computed within each perturbation replicate over the same 96 test images. ROC-AUC/PR-AUC bootstrap samples that contain only one class are skipped.

In [ ]:
def bootstrap_metric_ci(y,prob,pred,n_boot=BOOTSTRAP_N,seed=42):
    rng=np.random.default_rng(seed); n=len(y); vals={k:[] for k in ["accuracy","balanced_accuracy","sensitivity","specificity","f1","roc_auc","pr_auc"]}
    for _ in range(n_boot):
        idx=rng.integers(0,n,size=n); yb=y[idx]; pb=prob[idx]; db=pred[idx]
        if len(np.unique(yb))<2: continue
        tn,fp,fn,tp=confusion_matrix(yb,db,labels=[0,1]).ravel()
        vals["accuracy"].append(accuracy_score(yb,db)); vals["balanced_accuracy"].append(balanced_accuracy_score(yb,db)); vals["sensitivity"].append(recall_score(yb,db,zero_division=0)); vals["specificity"].append(tn/(tn+fp) if tn+fp else np.nan); vals["f1"].append(f1_score(yb,db,zero_division=0)); vals["roc_auc"].append(roc_auc_score(yb,pb)); vals["pr_auc"].append(average_precision_score(yb,pb))
    out={}
    for metric,v in vals.items():
        v=np.asarray(v,dtype=float); v=v[np.isfinite(v)]; out[f"{metric}_ci_low"]=np.quantile(v,.025); out[f"{metric}_ci_high"]=np.quantile(v,.975)
    return out
bootstrap_rows=[]
for _,row in metrics_df.iterrows():
    sub=predictions_df[(predictions_df["perturbation_level"]==row["perturbation_level"])&(predictions_df["repeat"]==row["repeat"])].sort_values("filename")
    y=sub["label"].to_numpy(int); prob=sub["predicted_probability_malignant"].to_numpy(float); pred=sub["predicted_label"].to_numpy(int)
    ci=bootstrap_metric_ci(y,prob,pred,BOOTSTRAP_N,SEED+int(row["repeat"])+int(row["perturbation_percent"])*100)
    bootstrap_rows.append({"perturbation_level":row["perturbation_level"],"perturbation_percent":row["perturbation_percent"],"repeat":row["repeat"],**ci})
bootstrap_df=pd.DataFrame(bootstrap_rows); bootstrap_df.to_csv(OUTPUT_DIR/"pass4_bootstrap_95ci_by_repeat.csv",index=False); display(bootstrap_df.round(4))

## 6. Publication figure: performance degradation

The primary robustness figure shows ROC-AUC, sensitivity, specificity, and F1 against localization perturbation. Error bars are ±1 SD across the five perturbation realizations.

In [ ]:
for mean_col,sd_col,ylabel,stem in [("roc_auc_mean","roc_auc_sd","ROC-AUC","roc_auc"),("sensitivity_mean","sensitivity_sd","Sensitivity","sensitivity"),("specificity_mean","specificity_sd","Specificity","specificity"),("f1_mean","f1_sd","F1 score","f1")]:
    plt.figure(figsize=(7,5)); x=summary_metrics["perturbation_percent"].to_numpy(); y=summary_metrics[mean_col].to_numpy(); sd=summary_metrics[sd_col].fillna(0).to_numpy()
    plt.errorbar(x,y,yerr=sd,marker="o",linewidth=2,capsize=4); plt.xlabel("Localization perturbation (%)"); plt.ylabel(ylabel); plt.xticks([0,5,10,15]); plt.ylim(0,1.02); plt.grid(alpha=.25); plt.tight_layout()
    path=FIG_DIR/f"pass4_{stem}.png"; plt.savefig(path,dpi=300,bbox_inches="tight"); plt.show(); print("Saved:",path)

## 7. Qualitative audit of perturbations

A benign and malignant test image are shown at 0%, 5%, 10%, and 15% perturbation to verify that the perturbations are visually plausible.

In [ ]:
examples=[test_df.index[test_df["label"]==0][0],test_df.index[test_df["label"]==1][0]]
for idx in examples:
    row=test_df.loc[idx]; image=load_rgb(row["image_resolved"]); mask=load_binary_mask(row["mask_path"],image.shape); dilated=dilate_mask(mask); ref=crop_box_from_mask(dilated)
    fig,axes=plt.subplots(1,4,figsize=(14,4))
    for j,level in enumerate(PERTURBATION_LEVELS):
        rng=np.random.default_rng(SEED+int(level*10000)); pbox=perturb_box(ref,image.shape,level,rng); roi=crop_masked_image(image,dilated,pbox); axes[j].imshow(roi); axes[j].set_title(f"{int(level*100)}% perturbation"); axes[j].axis("off")
    fig.suptitle(f"{row['class'].capitalize()} example: {row['filename']}"); plt.tight_layout(); path=FIG_DIR/f"pass4_example_{row['filename'].replace('.','_')}.png"; plt.savefig(path,dpi=300,bbox_inches="tight"); plt.show()

## 8. Final integrity checks

Pass 4 must use exactly the locked Pass 3B test cohort, the frozen raw-ROI checkpoint, a fixed 0.5 threshold, and no training.

In [ ]:
assert set(predictions_df["filename"])==set(test_df["filename"])
assert set(predictions_df["perturbation_level"].unique())==set(PERTURBATION_LEVELS)
for level in PERTURBATION_LEVELS:
    reps=predictions_df.loc[predictions_df["perturbation_level"]==level,"repeat"].unique()
    for repeat in reps:
        sub=predictions_df[(predictions_df["perturbation_level"]==level)&(predictions_df["repeat"]==repeat)]
        assert set(sub["filename"])==set(test_df["filename"])
zero=predictions_df[predictions_df["perturbation_level"]==0]
assert len(zero)==len(test_df) and np.allclose(zero["box_iou"],1.0)
integrity={"pass":"Pass 4","seed":SEED,"test_n":int(len(test_df)),"perturbation_levels_percent":[0,5,10,15],"nonzero_repeats_per_level":N_REPEATS,"bootstrap_resamples":BOOTSTRAP_N,"model_checkpoint":str(RAW_ROI_MODEL),"threshold":0.5,"retraining":False,"fine_tuning":False,"threshold_optimization":False,"zero_percent_reproduces_pass3b_raw_roi":True}
(OUTPUT_DIR/"pass4_integrity.json").write_text(json.dumps(integrity,indent=2)); print(json.dumps(integrity,indent=2)); print("PASS 4 COMPLETE")